In [1]:
## 1. IMPORT LIBRARIES
import pandas as pd
import re

In [2]:
## 2. LOAD EXCEL FILE
file_path = r"C:\Users\G_BOOTS\OneDrive\Desktop\heritage flowers\HERITAGE VARIETIES DAILY PRODUCTION RECORD.xlsx"
xls = pd.ExcelFile(file_path)
print(f"Loaded workbook with {len(xls.sheet_names)} sheets: {xls.sheet_names}")

Loaded workbook with 4 sheets: ['JANUARY 2026', 'FEBRUARY 2026', 'MARCH 2026', 'Sheet1']


3. DEFINE ROSE CATEGORIES
Quick lookup sets for classifying rose varieties

In [3]:
# Standard Roses (single stem varieties)
standard_roses = set([
    "MADAM RED","NIGHT GALE","ATHENA WHITE","ROYAL ATHENA","PINK ATHENA",
    "EVERED","REDBENTLY","BELLA ROSE","CARISE","ORANGE WAVE","MOONWALK",
    "MOVIE STAR","BAHAMA","CELINA","LABELLE","LAMPION","JUMILIA","CHELSIE",
    "DESERT QUEEN","REVIVAL","MADAM CERISE","MADAM PINK","PURPLE BLISS",
    "RED WAVE","PINK SWIRL","BALTTORE","THRILLER","MAITAI","TIGER","PIXIE","TRIALS"
])

# Spray Roses (multi-stem varieties)  
spray_roses = set([
    "WEDDING PARTY","ODILIA","SNW BUBBLE","SOWFLAKE","COLLEEN","SALINERO",
    "QUINCY","SNOW BLOSSOM","OLYMPIA","PUSKIN","ZADIQUE","PICANTO",
    "GOOD MOOD","SMOOTH.BLOSSOMS","YELLOW BABE","ROYAL MAGIC","FANCY",
    "MIRABELL","RTS","US","AZORE","VELOX","VELVET","GOLDEN BLOSSOM",
    "RELANX","INNOCENT","TARANTELLA","SWEET BLOSSOM","S.SHADOW","MARISSA",
    "B.BUBBLE","PASSION","EARTH AND FIRE","FIREWORK","LAND AND FIRE",
    "SEDINA","REFLEX","NIKITA","SPEAKING BLOSSOM","JEANINE"
])

In [4]:
## 4. CLASSIFY VARIETY FUNCTION
def get_category(variety):
    """Returns category: 'Standard Roses', 'Spray Roses', or 'Unknown'"""
    v = str(variety).upper().strip()
    if v in standard_roses:
        return "Standard Roses"
    if v in spray_roses:
        return "Spray Roses"
    return "Unknown"

In [5]:
# STANDARD ROSES
standard_roses = [
    "MADAM RED","NIGHT GALE","ATHENA WHITE","ROYAL ATHENA","PINK ATHENA",
    "EVERED","REDBENTLY","BELLA ROSE","CARISE","ORANGE WAVE","MOONWALK",
    "MOVIE STAR","BAHAMA","CELINA","LABELLE","LAMPION","JUMILIA","CHELSIE",
    "DESERT QUEEN","REVIVAL","MADAM CERISE","MADAM PINK","PURPLE BLISS",
    "RED WAVE","PINK SWIRL","BALTTORE","THRILLER","MAITAI","TIGER","PIXIE","TRIALS"
]

# SPRAY ROSES
spray_roses = [
    "WEDDING PARTY","ODILIA","SNW BUBBLE","SOWFLAKE","COLLEEN","SALINERO",
    "QUINCY","SNOW BLOSSOM","OLYMPIA","PUSKIN","ZADIQUE","PICANTO",
    "GOOD MOOD","SMOOTH.BLOSSOMS","YELLOW BABE","ROYAL MAGIC","FANCY",
    "MIRABELL","RTS","US","AZORE","VELOX","VELVET","GOLDEN BLOSSOM",
    "RELANX","INNOCENT","TARANTELLA","SWEET BLOSSOM","S.SHADOW","MARISSA",
    "B.BUBBLE","PASSION","EARTH AND FIRE","FIREWORK","LAND AND FIRE",
    "SEDINA","REFLEX","NIKITA","SPEAKING BLOSSOM"
]

In [6]:
## 4. CLASSIFY VARIETY FUNCTION
def get_category(variety):
    """Returns category: 'Standard Roses', 'Spray Roses', or 'Unknown'"""
    v = str(variety).upper().strip()
    if v in standard_roses:
        return "Standard Roses"
    if v in spray_roses:
        return "Spray Roses"
    return "Unknown"

5. DATE EXTRACTION FUNCTION
Handles 3 date formats across sheets:

Real datetime columns (Feb/March sheets)

Numeric days (Jan sheets: 1, 2, 3...)

String dates (modified Jan sheets)

In [7]:
def extract_date(sheet_name, col_value):
    """Smart date parser for different sheet formats"""
    
    # Case 1: Already a proper datetime
    if isinstance(col_value, pd.Timestamp):
        return col_value
    
    # Case 2: Numeric day (Jan sheets) → add month from sheet name
    if str(col_value).isdigit():
        try:
            day = int(col_value)
            month_map = {
                "JANUARY 2026": 1, "FEBRUARY 2026": 2, "MARCH 2026": 3
            }
            month = month_map.get(sheet_name.upper())
            if month:
                return pd.Timestamp(year=2026, month=month, day=day)
        except:
            pass
    
    # Case 3: String date → pandas parser
    parsed = pd.to_datetime(col_value, errors="coerce")
    if not pd.isna(parsed):
        return parsed
    
    return None

6. MAIN DATA EXTRACTION LOOP
Process each sheet → extract valid variety/production data

In [9]:
## 6. MAIN DATA EXTRACTION LOOP (FIXED)
clean_data = []

for sheet in xls.sheet_names:
    if "REPORT" in sheet.upper() or "SHEET1" in sheet.upper():
        continue
    
    print(f"📊 Processing: {sheet}")
    
    # Read and clean sheet
    df = pd.read_excel(xls, sheet_name=sheet)
    df = df.dropna(how="all").reset_index(drop=True)
    
    # Forward-fill variety column (first column)
    df.iloc[:, 0] = df.iloc[:, 0].ffill()
    
    # Process each row
    for i in range(len(df)):
        variety = str(df.iloc[i, 0]).strip().upper()
        
        # Skip invalid varieties
        if variety in ["NAN", "", "TOTAL"] or "ROSES" in variety:
            continue
        
        category = get_category(variety)
        
        # FIXED: Get column index instead of column name
        for col_idx in range(1, len(df.columns)):  # 👈 Start from column 1
            value = df.iloc[i, col_idx]  # 👈 Integer index!
            
            if pd.isna(value):
                continue
            
            # Get column name for date extraction
            col_name = df.columns[col_idx]
            date = extract_date(sheet, col_name)
            
            if date is None:
                continue
            
            clean_data.append({
                "Date": date,
                "Variety": variety,
                "Category": category,
                "Production_Qty": value
            })

print(f"\n✅ Extracted {len(clean_data)} raw records")

📊 Processing: JANUARY 2026
📊 Processing: FEBRUARY 2026
📊 Processing: MARCH 2026

✅ Extracted 4379 raw records


7. FINAL CLEANING & EXPORT
Convert types → filter junk → sort → save

In [11]:
# Create final DataFrame
clean_df = pd.DataFrame(clean_data)

if clean_df.empty:
    print("❌ No data extracted — check file structure")
else:
    # Clean numeric column
    clean_df["Production_Qty"] = pd.to_numeric(clean_df["Production_Qty"], errors="coerce")
    
    # Filter valid data only
    clean_df = clean_df.dropna(subset=["Production_Qty"])
    clean_df = clean_df[clean_df["Production_Qty"] > 0]
    
    # Sort chronologically
    clean_df = clean_df.sort_values(["Date", "Variety"]).reset_index(drop=True)
    
    # Export
    output_path = r"C:\Users\G_BOOTS\OneDrive\Desktop\heritage flowers\clean_production_data.xlsx"
    clean_df.to_excel(output_path, index=False)
    
    # Summary stats
    print("\n🎉 CLEANING COMPLETE!")
    print(f"📅 Date Range: {clean_df['Date'].min()} → {clean_df['Date'].max()}")
    print(f"📈 Total Rows: {len(clean_df):,}")
    print("\nCategory Breakdown:")
    print(clean_df["Category"].value_counts())
    
    # Quick preview
    print("\n🔍 Sample data:")
    print(clean_df.head())


🎉 CLEANING COMPLETE!
📅 Date Range: 2026-01-01 00:00:00 → 2026-03-04 00:00:00
📈 Total Rows: 3,268

Category Breakdown:
Category
Spray Roses       1657
Standard Roses    1609
Unknown              2
Name: count, dtype: int64

🔍 Sample data:
        Date       Variety        Category  Production_Qty
0 2026-01-01  ATHENA WHITE  Standard Roses          4620.0
1 2026-01-01         AZORE     Spray Roses            10.0
2 2026-01-01        BAHAMA  Standard Roses           246.0
3 2026-01-01    BELLA ROSE  Standard Roses          1650.0
4 2026-01-01        CARISE  Standard Roses            95.0
